# Tesis v4 · Etapa A: FairFinGAN por seed
Runtime **con GPU** (Entorno de ejecución → Cambiar tipo → T4) y **Ejecutar todo**. Corre las 10 seeds y guarda `synth_seedN.csv` en `Drive/tesis_backup/v4`. Si se corta, volver a ejecutar todo: las seeds ya hechas se saltean.

Cambios respecto de la corrida anterior: el GAN ya no recibe `ID` ni `AGE_GROUP`, y `PAY_*` se tratan como categóricas. El código de FairFinGAN (3 celdas) está sin modificar.

In [ ]:
# ===== CELDA 0 · SETUP (correr primero, en cualquiera de los dos notebooks) =====
import os, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
warnings.filterwarnings('ignore')

# --- carpeta de salida en Drive (v4 = corrida con CV, carpeta limpia) ---
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/tesis_backup/v4'
os.makedirs(OUT, exist_ok=True)

SEEDS = list(range(1, 11))
target_col = 'default.payment.next.month'
drop_cols = ['ID', 'SEX', target_col, 'AGE_GROUP']

# --- carga y limpieza (idéntico a Bozza tesis) ---
url = "https://raw.githubusercontent.com/YuChenAmberLu/Data-Science--Credit-Card-Default/master/UCI_Credit_Card.csv"
df = pd.read_csv(url)
df_clean = df.copy()
df_clean['EDUCATION'] = df_clean['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df_clean['MARRIAGE'] = df_clean['MARRIAGE'].replace({0: 3})
pay_cols = [c for c in df_clean.columns if c.startswith('PAY_') and 'AMT' not in c]
for c in pay_cols:
    df_clean[c] = df_clean[c].replace({-2: -1, 0: -1})
df_clean['AGE_GROUP'] = df_clean['AGE'].apply(lambda x: 'core' if 25 <= x <= 65 else 'edge')

# --- inyección de sesgo (idéntico a Bozza tesis: una sola vez, seed 42, antes del split) ---
def inject_bias(df, sex_col='SEX', target_col='default.payment.next.month',
                underpriv_code=1, priv_code=2,
                flip_frac_underpriv=0.15, flip_frac_priv=0.15, seed=42):
    rng = np.random.default_rng(seed)
    df2 = df.copy()
    mask_u = (df2[sex_col] == underpriv_code) & (df2[target_col] == 0)
    idx_u = df2[mask_u].index
    flip_idx_u = rng.choice(idx_u, size=int(len(idx_u) * flip_frac_underpriv), replace=False)
    df2.loc[flip_idx_u, target_col] = 1
    mask_p = (df2[sex_col] == priv_code) & (df2[target_col] == 1)
    idx_p = df2[mask_p].index
    flip_idx_p = rng.choice(idx_p, size=int(len(idx_p) * flip_frac_priv), replace=False)
    df2.loc[flip_idx_p, target_col] = 0
    return df2

df_biased = inject_bias(df_clean, underpriv_code=1, priv_code=2)
df_biased['AGE_GROUP'] = pd.factorize(df_biased['AGE_GROUP'])[0]

def fairness_metrics(y_true, y_pred, sensitive, priv_val, underpriv_val):
    def rates(mask):
        yt = np.asarray(y_true)[mask]; yp = np.asarray(y_pred)[mask]
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        ppr = (yp == 1).mean()
        return tpr, fpr, ppr
    sensitive = np.asarray(sensitive)
    tpr_p, fpr_p, ppr_p = rates(sensitive == priv_val)
    tpr_u, fpr_u, ppr_u = rates(sensitive == underpriv_val)
    return {
        'statistical_parity_diff': ppr_p - ppr_u,
        'TPR_priv (women)': tpr_p, 'TPR_underpriv (men)': tpr_u, 'TPR_diff (EOd)': tpr_p - tpr_u,
        'FPR_priv (women)': fpr_p, 'FPR_underpriv (men)': fpr_u, 'FPR_diff (EOd)': fpr_p - fpr_u,
    }

def split(seed):
    """Mismo split que la corrida anterior."""
    return train_test_split(df_biased, test_size=0.2, random_state=seed,
                            stratify=df_biased[target_col])

def save_csv_atomic(d, path, **kw):
    """Guarda en .tmp y renombra: nunca queda un archivo a medio escribir."""
    d.to_csv(path + '.tmp', **kw)
    os.replace(path + '.tmp', path)

print("Setup OK", df_biased.shape)
print("Default rate por sexo (original):", df_clean.groupby('SEX')[target_col].mean().round(4).to_dict())
print("Default rate por sexo (con sesgo):", df_biased.groupby('SEX')[target_col].mean().round(4).to_dict())
print("Default rate por edad, original (core = 25-65, edge = resto):",
      df_clean.groupby('AGE_GROUP')[target_col].mean().round(4).to_dict(), "| tamaños:", df_clean['AGE_GROUP'].value_counts().to_dict())


In [ ]:
#Attribution
# ============================================================
# The code in cells 7, 8 and 9 is from FairFinGAN (Le Quy et al., 2026),
# forked from https://github.com/tailequy/FairFinGAN — used as-is,
# unmodified, as the synthetic data generation engine.
# Only cell 10 (Generating) was adapted by me.
# ============================================================

#Utils
import torch
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
from collections import OrderedDict
from typing import Tuple, List, Dict, Union, Optional
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer
from sklearn.model_selection import train_test_split

def get_cont_cat_transform(df):
    df_int = df.select_dtypes(['float', 'integer']).values
    continuous_columns_list = list(df.select_dtypes(['float', 'integer']).columns)
    scaler = QuantileTransformer(n_quantiles=2000, output_distribution='uniform')
    df_int = scaler.fit_transform(df_int)
    df_cat = df.select_dtypes('object')
    df_cat_names = list(df.select_dtypes('object').columns)
    numerical_array = df_int
    ohe = OneHotEncoder()
    ohe_array = ohe.fit_transform(df_cat)
    cat_lens = [i.shape[0] for i in ohe.categories_]
    discrete_columns_ordereddict = OrderedDict(zip(df_cat_names, cat_lens))
    return ohe, scaler, discrete_columns_ordereddict, continuous_columns_list, numerical_array, ohe_array

def get_ohe_data_fair(df, S, Y, S_under, Y_desire):
    ohe, scaler, discrete_columns_ordereddict, continuous_columns_list, numerical_array, ohe_array = get_cont_cat_transform(df)
    S_start_index = len(continuous_columns_list) + sum(
        list(discrete_columns_ordereddict.values())[:list(discrete_columns_ordereddict.keys()).index(S)])
    Y_start_index = len(continuous_columns_list) + sum(
        list(discrete_columns_ordereddict.values())[:list(discrete_columns_ordereddict.keys()).index(Y)])
    if ohe.categories_[list(discrete_columns_ordereddict.keys()).index(S)][0] == S_under:
        underpriv_index, priv_index = 0, 1
    else:
        underpriv_index, priv_index = 1, 0
    if ohe.categories_[list(discrete_columns_ordereddict.keys()).index(Y)][0] == Y_desire:
        desire_index, undesire_index = 0, 1
    else:
        desire_index, undesire_index = 1, 0
    final_array = np.hstack((numerical_array, ohe_array.toarray()))
    return ohe, scaler, discrete_columns_ordereddict, continuous_columns_list, final_array, S_start_index, Y_start_index, underpriv_index, priv_index, undesire_index, desire_index

def get_original_data(df_transformed, df_orig, ohe, scaler):
    df_ohe_int = df_transformed[:, :df_orig.select_dtypes(['float', 'integer']).shape[1]]
    df_ohe_int = scaler.inverse_transform(df_ohe_int)
    df_ohe_cats = df_transformed[:, df_orig.select_dtypes(['float', 'integer']).shape[1]:]
    df_ohe_cats = ohe.inverse_transform(df_ohe_cats)
    df_int = pd.DataFrame(df_ohe_int, columns=df_orig.select_dtypes(['float', 'integer']).columns)
    df_cat = pd.DataFrame(df_ohe_cats, columns=df_orig.select_dtypes('object').columns)
    return pd.concat([df_int, df_cat], axis=1)

def prepare_data_fair(df, batch_size, S, Y, S_under, Y_desire):
    ohe, scaler, discrete_columns, continuous_columns, df_transformed, S_start_index, Y_start_index, underpriv_index, priv_index, undesire_index, desire_index = get_ohe_data_fair(df, S, Y, S_under, Y_desire)
    input_dim = df_transformed.shape[1]
    X_train, X_test = train_test_split(df_transformed, test_size=0.1, shuffle=True)
    data_train, data_test = X_train.copy(), X_test.copy()
    data = torch.from_numpy(data_train).float()
    train_ds = TensorDataset(data)
    train_dl = DataLoader(train_ds, batch_size=batch_size, drop_last=True)
    return ohe, scaler, input_dim, discrete_columns, continuous_columns, train_dl, data_train, data_test, S_start_index, Y_start_index, underpriv_index, priv_index, undesire_index, desire_index

assert callable(prepare_data_fair), "prepare_data_fair was not defined — check indentation"
print("Cell 7 OK, all functions defined correctly")



In [ ]:
#GANModules + Classifier:
import torch
import torch.nn.functional as f
from torch import nn

class Generator(nn.Module):
    def __init__(self, input_dim, continuous_columns, discrete_columns):
        super(Generator, self).__init__()
        self._input_dim = input_dim
        self._discrete_columns = discrete_columns
        self._num_continuous_columns = len(continuous_columns)
        self.lin1 = nn.Linear(self._input_dim, self._input_dim)
        self.lin_numerical = nn.Linear(self._input_dim, self._num_continuous_columns)
        self.lin_cat = nn.ModuleDict()
        for key, value in self._discrete_columns.items():
            self.lin_cat[key] = nn.Linear(self._input_dim, value)

    def forward(self, x):
        x = torch.relu(self.lin1(x))
        x_numerical = f.relu(self.lin_numerical(x))
        x_cat = []
        for key in self.lin_cat:
            x_cat.append(f.gumbel_softmax(self.lin_cat[key](x), tau=0.2))
        x_final = torch.cat((x_numerical, *x_cat), 1)
        return x_final

class Critic(nn.Module):
    def __init__(self, input_dim):
        super(Critic, self).__init__()
        self._input_dim = input_dim
        self.dense1 = nn.Linear(self._input_dim, self._input_dim)
        self.dense2 = nn.Linear(self._input_dim, self._input_dim)

    def forward(self, x):
        x = f.leaky_relu(self.dense1(x))
        x = f.leaky_relu(self.dense2(x))
        return x

class FairLossFunc(nn.Module):
    def __init__(self, S_start_index, Y_start_index, underpriv_index, priv_index, undesire_index, desire_index, classifier):
        super(FairLossFunc, self).__init__()
        self._S_start_index = S_start_index
        self._Y_start_index = Y_start_index
        self._underpriv_index = underpriv_index
        self._priv_index = priv_index
        self._undesire_index = undesire_index
        self._desire_index = desire_index
        self._classifier = classifier

    def forward(self, x, crit_fake_pred, lamda):
        G = x[:, self._S_start_index:self._S_start_index + 2]
        x_h = torch.cat((x[:, :self._Y_start_index], x[:, self._Y_start_index + 2:]), dim=1)
        logits = self._classifier(x_h)
        I = torch.nn.functional.gumbel_softmax(logits, tau=0.2, hard=False)
        disp = -1.0 * lamda * (torch.mean(G[:, self._underpriv_index] * I[:, self._desire_index]) / (
            x[:, self._S_start_index + self._underpriv_index].sum()) - torch.mean(
            G[:, self._priv_index] * I[:, self._desire_index]) / (
                                   x[:, self._S_start_index + self._priv_index].sum())) - 1.0 * torch.mean(crit_fake_pred)
        return disp

def get_gradient(crit, real, fake, epsilon):
    mixed_data = real * epsilon + fake * (1 - epsilon)
    mixed_scores = crit(mixed_data)
    gradient = torch.autograd.grad(
        inputs=mixed_data, outputs=mixed_scores,
        grad_outputs=torch.ones_like(mixed_scores),
        create_graph=True, retain_graph=True,
    )[0]
    return gradient

def gradient_penalty(gradient):
    gradient = gradient.view(len(gradient), -1)
    gradient_norm = gradient.norm(2, dim=1)
    penalty = torch.mean((gradient_norm - 1) ** 2)
    return penalty

def get_gen_loss(crit_fake_pred):
    return -1. * torch.mean(crit_fake_pred)

def get_crit_loss(crit_fake_pred, crit_real_pred, gp, c_lambda):
    return torch.mean(crit_fake_pred) - torch.mean(crit_real_pred) + c_lambda * gp

class ClassifierMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64)):
        super(ClassifierMLP, self).__init__()
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden_dims[0]),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dims[0], hidden_dims[1]),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dims[1], 2),
        )

    def forward(self, x):
        return self.mlp(x)

    def train_model(self, train_loader, num_epochs=50, lr=0.001):
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        for epoch in range(num_epochs):
            self.train()
            total_loss = 0.0
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = self(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            avg_loss = total_loss / len(train_loader)
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

    def evaluate_model(self, test_loader):
        self.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = self(batch_X)
                _, predicted = torch.max(outputs, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        print(f"Test Accuracy: {correct / total * 100:.2f}%")
        return correct / total



In [ ]:
#Training:
import torch
import torch.nn.functional as f
from torch import nn
import pandas as pd
import numpy as np
from collections import OrderedDict
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer
from sklearn.model_selection import train_test_split
import argparse
from tqdm.auto import tqdm

display_step = 50

class TFG:
    def __init__(self, df, epochs, batch_size, device=None, fairness_config=None):
        self.df = df
        self.epochs = epochs
        self.batch_size = batch_size
        self.device = "cpu" if device is None else device
        self.fair_epochs = fairness_config.get('fair_epochs', 0) if fairness_config else 0
        if fairness_config and self.fair_epochs > 0:
            self.lamda = fairness_config.get('lamda')
            self.S = fairness_config.get('S')
            self.Y = fairness_config.get('Y')
            self.S_under = fairness_config.get('S_under')
            self.Y_desire = fairness_config.get('Y_desire')
        else:
            self.lamda = None; self.S = None; self.Y = None; self.S_under = None; self.Y_desire = None

    def prepare_data(self):
        if self.fair_epochs > 0:
            self.ohe, self.scaler, self.input_dim, self.discrete_columns, self.continuous_columns, self.train_dl, self.data_train, self.data_test, self.S_start_index, self.Y_start_index, self.underpriv_index, self.priv_index, self.undesire_index, self.desire_index = prepare_data_fair(self.df, self.batch_size, self.S, self.Y, self.S_under, self.Y_desire)
        else:
            self.ohe, self.scaler, self.input_dim, self.discrete_columns, self.continuous_columns, self.train_dl, self.data_train, self.data_test = prepare_data_nofair(self.df, self.batch_size)

    def train(self):
        self.prepare_data()
        self.simpleClassifier = ClassifierMLP(input_dim=self.input_dim - 2)
        X_real = pd.DataFrame(self.data_train).iloc[:, :-2].values
        y_real = pd.DataFrame(self.data_train).iloc[:, -2:].values
        y_real = np.argmax(y_real, axis=1)
        Xtr_real, Xts_real, Ytr_real, Yts_real = train_test_split(X_real, y_real, test_size=0.2, random_state=42)
        print(Xtr_real.shape)
        Xtr_tensor = torch.tensor(Xtr_real, dtype=torch.float32)
        Ytr_tensor = torch.tensor(Ytr_real, dtype=torch.long)
        Xts_tensor = torch.tensor(Xts_real, dtype=torch.float32)
        Yts_tensor = torch.tensor(Yts_real, dtype=torch.long)
        train_dataset = TensorDataset(Xtr_tensor, Ytr_tensor)
        test_dataset = TensorDataset(Xts_tensor, Yts_tensor)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)
        test_loader = DataLoader(test_dataset, batch_size=32)
        self.simpleClassifier.train_model(train_loader=train_loader)
        self.simpleClassifier.evaluate_model(test_loader=test_loader)
        self.generator = Generator(self.input_dim, self.continuous_columns, self.discrete_columns).to(self.device)
        self.critic = Critic(self.input_dim).to(self.device)
        if self.fair_epochs > 0:
            self.second_critic = FairLossFunc(self.S_start_index, self.Y_start_index, self.underpriv_index, self.priv_index, self.undesire_index, self.desire_index, self.simpleClassifier).to(self.device)
        self.gen_optimizer = torch.optim.Adam(self.generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
        self.gen_optimizer_fair = torch.optim.Adam(self.generator.parameters(), lr=0.0001, betas=(0.5, 0.999))
        self.crit_optimizer = torch.optim.Adam(self.critic.parameters(), lr=0.0002, betas=(0.5, 0.999))
        critic_losses = []
        cur_step = 0
        with tqdm(total=self.epochs, desc="Training Progress", ncols=100) as pbar:
            for i in range(self.epochs):
                print("epoch {}".format(i + 1))
                if i + 1 <= (self.epochs - self.fair_epochs):
                    pbar.set_postfix_str("Training for accuracy")
                if i + 1 > (self.epochs - self.fair_epochs):
                    pbar.set_postfix_str("Training for fairness")
                for data in self.train_dl:
                    data[0] = data[0].to(self.device)
                    crit_repeat = 4
                    mean_iteration_critic_loss = 0
                    for k in range(crit_repeat):
                        self.crit_optimizer.zero_grad()
                        fake_noise = torch.randn(size=(self.batch_size, self.input_dim), device=self.device).float()
                        fake = self.generator(fake_noise)
                        crit_fake_pred = self.critic(fake.detach())
                        crit_real_pred = self.critic(data[0])
                        epsilon = torch.rand(self.batch_size, self.input_dim, device=self.device, requires_grad=True)
                        gradient = get_gradient(self.critic, data[0], fake.detach(), epsilon)
                        gp = gradient_penalty(gradient)
                        crit_loss = get_crit_loss(crit_fake_pred, crit_real_pred, gp, c_lambda=10)
                        mean_iteration_critic_loss += crit_loss.item() / crit_repeat
                        crit_loss.backward(retain_graph=True)
                        self.crit_optimizer.step()
                    if cur_step > 50:
                        critic_losses += [mean_iteration_critic_loss]
                    if i + 1 <= (self.epochs - self.fair_epochs):
                        self.gen_optimizer.zero_grad()
                        fake_noise_2 = torch.randn(size=(self.batch_size, self.input_dim), device=self.device).float()
                        fake_2 = self.generator(fake_noise_2)
                        crit_fake_pred = self.critic(fake_2)
                        gen_loss = get_gen_loss(crit_fake_pred)
                        gen_loss.backward()
                        self.gen_optimizer.step()
                    if i + 1 > (self.epochs - self.fair_epochs):
                        self.gen_optimizer_fair.zero_grad()
                        fake_noise_2 = torch.randn(size=(self.batch_size, self.input_dim), device=self.device).float()
                        fake_2 = self.generator(fake_noise_2)
                        crit_fake_pred = self.critic(fake_2)
                        gen_fair_loss = self.second_critic(fake_2, crit_fake_pred, self.lamda)
                        gen_fair_loss.backward()
                        self.gen_optimizer_fair.step()
                    cur_step += 1
                pbar.update(1)

    def generate_fake_df(self, num_rows):
        with torch.no_grad():
            fake_numpy_array = self.generator(torch.randn(size=(num_rows, self.input_dim), device=self.device)).cpu().detach().numpy()
            fake_df = get_original_data(fake_numpy_array, self.df, self.ohe, self.scaler)
            return fake_df[self.df.columns]



In [ ]:
# ===== CELDA A · ETAPA A: FairFinGAN por seed (runtime con GPU) =====
# Antes: Celda 0 + las 3 celdas de FairFinGAN de arriba (código de los autores, sin modificar).
# Si Colab se corta, volvé a correr todo: las seeds ya guardadas se saltean.
# OJO: no corras en esta sesión las celdas viejas de gráficos (usan "f" como variable y pisan torch.nn.functional).
import torch
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)

fairness_config = {'fair_epochs': 100, 'lamda': 1.0,
                   'S': 'SEX', 'Y': 'default', 'S_under': '1', 'Y_desire': '0'}

# Variables categóricas para el GAN. PAY_* son estados discretos de pago (-1, 1, 2, ...):
# como categóricas el GAN solo puede generar valores válidos (antes salían cosas como 1.6).
CAT_GAN = ['SEX', 'EDUCATION', 'MARRIAGE'] + pay_cols + ['default']

def make_gan_input(df_train_real):
    # ID y AGE_GROUP fuera: ID no es información y AGE_GROUP se deriva de AGE
    d = df_train_real.drop(columns=['ID', 'AGE_GROUP']).rename(columns={target_col: 'default'})
    for col in CAT_GAN:
        d[col] = d[col].astype(str)
    cat = list(d.select_dtypes('object').columns)
    assert cat[-1] == 'default', f"FairFinGAN necesita el target como última categórica, quedó {cat[-1]}"
    return d

for seed in SEEDS:
    path = f'{OUT}/synth_seed{seed}.csv'
    if os.path.exists(path):
        print(f'seed {seed}: ya existe, la salteo')
        continue
    t0 = time.time()
    torch.manual_seed(seed)
    np.random.seed(seed)

    df_train_real, df_test_real = split(seed)
    df_gan_s = make_gan_input(df_train_real)

    tfg_s = TFG(df_gan_s, epochs=200, batch_size=256, device=device, fairness_config=fairness_config)
    tfg_s.train()
    synth = tfg_s.generate_fake_df(len(df_gan_s))
    for col in CAT_GAN:
        synth[col] = synth[col].astype(int)

    save_csv_atomic(pd.Series(df_test_real['ID'].values, name='ID').to_frame(), f'{OUT}/test_ids_seed{seed}.csv', index=False)
    save_csv_atomic(synth, path, index=False)   # la etapa B solo lo ve cuando está completo
    print(f'===== seed {seed} lista en {(time.time() - t0) / 60:.1f} min =====')

print("Etapa A terminada.")
